In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
file_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming/bmw_vehicle_new.csv"
new_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

print("Incoming Records:", new_df.count())

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

merge_df = (
    new_df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", to_date("purchase_date"))
    .withColumn("vin", upper(trim(col("vin"))))
    .withColumn("model", trim(col("model")))
    .withColumn("fuel_type", upper(trim(col("fuel_type"))))
    .withColumn("processed_at", current_timestamp())
)


target = DeltaTable.forName(
    spark,
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)


(
    target.alias("target")
    .merge(
        merge_df.alias("source"),
        "target.vehicle_id = source.vehicle_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Delta MERGE completed successfully!")

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Clean and prepare incoming records
merge_df = (
    new_df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", to_date("purchase_date"))
    .withColumn("vin", upper(trim(col("vin"))))
    .withColumn("model", trim(col("model")))
    .withColumn("fuel_type", upper(trim(col("fuel_type"))))
    .withColumn("processed_at", current_timestamp())
)

# Connect to existing Silver table
target = DeltaTable.forName(
    spark,
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)

# Perform MERGE
(
    target.alias("target")
    .merge(
        merge_df.alias("source"),
        "target.vehicle_id = source.vehicle_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Delta MERGE completed successfully!")

In [0]:
# 1. Vehicle Summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.vehicle_summary AS
SELECT
    model,
    fuel_type,
    COUNT(*) AS total_vehicles,
    ROUND(AVG(model_year), 2) AS average_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model, fuel_type
""")

# 2. Model Summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.model_summary AS
SELECT
    model,
    COUNT(*) AS total_vehicles,
    MIN(model_year) AS oldest_model_year,
    MAX(model_year) AS newest_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model
""")

# 3. Fuel Summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.fuel_summary AS
SELECT
    fuel_type,
    COUNT(*) AS total_vehicles,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY fuel_type
""")

# 4. Purchase Year Summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.purchase_year_summary AS
SELECT
    YEAR(purchase_date) AS purchase_year,
    COUNT(*) AS total_purchases
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY YEAR(purchase_date)
""")

print("All Gold tables refreshed successfully!")